### Project 1

Every month, a new trendy tea or coffee shop seems to appear somewhere in New York City, often attracting lines that stretch for multiple blocks. Many young New Yorkers, including myself, visit these establishments to enjoy a variety of unique beverages inspired by flavors and traditions from around the world. Popular chains such as HeyTea, Luckin Coffee, Blue Bottle Coffee, and Tiger Sugar offer drinks ranging from boba tea and matcha-based beverages to innovative coffee creations.
Many New Yorkers can easily discuss their favorite tea or coffee shop based on factors such as price, drink selection, quality, and popularity. This recommendation system is designed to help New Yorkers discover new tea and coffee destinations that may be worth waiting in a long line to experience.
In this study, six New Yorkers will rate their favorite tea or coffee locations from a list of five featured establishments. The featured locations are HeyTea (HT), Blue Bottle Coffee (BB), Luckin Coffee (LC), Tiger Sugar (TS), and Nana's Green Tea (NG).

In [49]:
# import library 
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split

In [50]:
# Random seed to repeat results
np.random.seed(808) 

# Names of New Yorkers that will provide the ratings
new_yorker = ['Nana', 'John', 'Big Mike', 'Brunson', 'Lisa', 'Marge']

# List of abbreviations for the five coffee/tea shops.
# HT = HeyTea
# BB = Blue Bottle Coffee
# LC = Luckin Coffee
# TS = Tiger Sugar
# NG = Hungry Ghost Coffee
spots = ['HT', 'BB','LC','TS','NG']

# Generate a 6 x 5 matrix of random integers from 1 to 10.
# Each row represents a New Yorker.
# Each column represents a coffee/tea shop.
ratings = np.random.randint(1,11, size = (6,5)).astype(float)

# condtion to ensure 30 percent of the data is missing 
mask = np.random.rand(6,5) < 0.3
ratings[mask] = np.nan 

# Convert NumPy array to Pandas Dataframe
# Rows are labeld with New Yoker names.
# Columns are labeld with coffee/tea shop
df = pd.DataFrame(ratings, index = new_yorker, columns = spots)

In [51]:
print(df)

            HT   BB   LC    TS   NG
Nana       6.0  NaN  6.0  10.0  NaN
John       8.0  5.0  6.0   7.0  7.0
Big Mike   NaN  4.0  9.0   NaN  2.0
Brunson   10.0  NaN  9.0   6.0  NaN
Lisa       NaN  6.0  7.0   3.0  NaN
Marge      9.0  4.0  NaN   9.0  7.0


In [52]:
# Create a list of all existing ratings in the matrix.
# Each tuple (i, j) represents:
# i = New Yorker  in rows
# j = Coffee/tea location in columns

rating_indices = [
    (i, j)

    # Loop through all New Yorkers (rows)
    for i in range(df.shape[0])

    # Loop through all coffee/tea locations (columns)
    for j in range(df.shape[1])

    # Keep only ratings that are not missing (not NaN)
    if not np.isnan(df.iat[i, j])
]

### Trainng and Test Dataset

In [53]:
# Split the available coffee/tea ratings into training and testing sets.
# 80% of the ratings will be used to teach the recommendation system
# about user preferences, while 20% will be held out to evaluate
# how accurately the system can predict unseen ratings.

# A random seed is used so the same train-test split can be reproduced.
train_indices, test_indices = train_test_split(
    rating_indices,
    test_size=0.2,
    random_state= 909
)

# Create empty training and testing matrices with the same dimensions
# as the original ratings matrix. Each row represents a New Yorker
# and each column represents a coffee or tea location.

# All values are initially set to NaN, creating a blank matrix that
# will later be populated with either training or testing ratings.

# Separating the data in this way prevents information from the test set
# from influencing the learning process, allowing for a fair evaluation
# of the recommendation system's performance.
train = pd.DataFrame(np.nan, index=new_yorker, columns=spots)
test = pd.DataFrame(np.nan, index=new_yorker, columns=spots)

# Fill the training matrix with the ratings selected for model training.
# These ratings are the only information the recommendation system
# will use to learn user preferences.
for (i, j) in train_indices:
    train.iat[i, j] = df.iat[i, j]

# Fill the testing matrix with the ratings held out for evaluation.
# After training, the model will attempt to predict these ratings,
# and the predictions will be compared to the actual values to measure accuracy.
for (i, j) in test_indices:
    test.iat[i, j] = df.iat[i, j]

### Raw Average
The raw average is the overall mean of all the rating in the training dataset, representing the general rating given by New Yorkers to coffee and tea spots. It is calculated by averaging all availble ratings while ignoring missing values. 

In [54]:
# Display the training matrix containing the ratings that will be used
# to teach the recommendation system about New Yorkers' preferences.
print(train)

# The .stack() function removes all missing values (NaN) and converts
# The matrix into a single list of observed ratings are collected while missing ratings are ignored.

# The .mean() function then calculates the average rating across all
# coffee and tea locations in the training data.
# This average serves as a simple baseline recommendation. 
raw_avg = train.stack().mean()
# We can compare our recommendation model against this
#  baseline to see whether the model provides more accurate predictions.

# Display the overall average rating from the training data.
print(f"\nRaw Average for Training Data: {raw_avg}")

            HT   BB   LC    TS   NG
Nana       6.0  NaN  6.0  10.0  NaN
John       8.0  5.0  6.0   7.0  7.0
Big Mike   NaN  4.0  9.0   NaN  NaN
Brunson   10.0  NaN  NaN   NaN  NaN
Lisa       NaN  6.0  7.0   NaN  NaN
Marge      NaN  4.0  NaN   9.0  7.0

Raw Average for Training Data: 6.9375


### Root Mean Squared Error For Training and Test Data

Root Mean Squared Error (RMSE) is used to measure how accurate the recommendation system is at predicting New Yorkers’ ratings for coffee and tea locations. The training RMSE shows how well the model fits the ratings it was trained on, while the test RMSE evaluates how well it predicts unseen ratings. Lower RMSE values indicate more accurate predictions, and comparing both helps determine whether the model generalizes well or is overfitting the training data.



In [55]:
# This function compares predicted ratings with actual ratings,
# calculates the prediction errors, squares them, averages them,
# and then takes the square root.

# np.nanmean() is used so that any missing ratings (NaN values)
# are ignored during the calculation.
def rmse(predictions, targets):
    return np.sqrt(np.nanmean((predictions - targets) ** 2))

# Calculate the RMSE for the training data.

# Extract only the ratings that exist in the training matrix,
# excluding any missing ratings.

# For the baseline model, every coffee and tea location is predicted
# using the overall average rating from the training data (raw_avg).

# np.full_like() creates an array with the same shape as the observed
# ratings and fills every value with the training average rating.
train_true = train.values[~np.isnan(train.values)]
train_pred = np.full_like(train_true, raw_avg)
train_rmse = rmse(train_pred, train_true)

# Calculate the RMSE for the testing data.

# Use the same baseline prediction (the overall training average)
# for every held-out rating and compare those predictions to the
# actual ratings given by New Yorkers.

# This allows us to evaluate how well the baseline model predicts
# unseen coffee and tea shop preferences.
test_true = test.values[~np.isnan(test.values)]
test_pred = np.full_like(test_true, raw_avg)
test_rmse = rmse(test_pred, test_true)


# Lower RMSE values indicate that the predictions are closer to the 
# actual ratings and therefore more accurate.
print(f"\nTrain Data RMSE: {train_rmse}")
print(f"\nTest Data RMSE: {test_rmse}")


Train Data RMSE: 1.818954026356906

Test Data RMSE: 3.1390932209795874


### User and Location Bias
The user bias is the average difference between a New Yorker's ratings and the overall average rating across all coffee and tea locations. The location bias is the average difference between a location's ratings and the ratings expected after accounting for user biases. These biases help identify users who tend to rate more generously or critically and locations that are consistently rated higher or lower than expected.

In [56]:
# Calculate the user bias by finding each New Yorker's
# average rating and comparing it to the overall average rating.

# Calculate the location bias by finding each coffee/tea spot's
# average rating and comparing it to the overall average rating.

# Positive values indicate ratings above average,
# while negative values indicate ratings below average.
user_bias = train.mean(axis=1) - raw_avg
item_bias = train.mean(axis=0) - raw_avg

print(f"New Yorker User Bias:\n{user_bias}")
print(f"\nCoffee/Tea Location Bias:\n{item_bias}")

New Yorker User Bias:
Nana        0.395833
John       -0.337500
Big Mike   -0.437500
Brunson     3.062500
Lisa       -0.437500
Marge      -0.270833
dtype: float64

Coffee/Tea Location Bias:
HT    1.062500
BB   -2.187500
LC    0.062500
TS    1.729167
NG    0.062500
dtype: float64


### Baseline Predictors
The baseline predictor estimates ratings using the overall average rating, user bias, and location bias. 

In [57]:
# Create a DataFrame to store baseline predicted ratings
# for each New Yorker and coffee/tea location.
baseline_predictions = pd.DataFrame(index=new_yorker, columns=spots)

# Calculate a baseline prediction for every
# New Yorker-location combination.

# Prediction = Overall Average Rating
#            + New Yorker Bias
#            + Location Bias

for user in new_yorker:
    for spot in spots:
        baseline_predictions.at[user, spot] = (
            raw_avg
            + user_bias[user]
            + item_bias[spot]
        )

print("Baseline Predictions for New Yorkers and Coffee/Tea Locations:")
print(baseline_predictions)

Baseline Predictions for New Yorkers and Coffee/Tea Locations:
                HT        BB        LC         TS        NG
Nana      8.395833  5.145833  7.395833     9.0625  7.395833
John        7.6625    4.4125    6.6625   8.329167    6.6625
Big Mike    7.5625    4.3125    6.5625   8.229167    6.5625
Brunson    11.0625    7.8125   10.0625  11.729167   10.0625
Lisa        7.5625    4.3125    6.5625   8.229167    6.5625
Marge     7.729167  4.479167  6.729167   8.395833  6.729167


### RMSE for baseline predictors on train and test data
 Here we measure how accurately the baseline predictor estimates New Yorkers' ratings for coffee and tea locations. The baseline model uses the overall average rating along with user and location biases to make predictions. Training RMSE evaluates prediction accuracy on the ratings used to build the model, while testing RMSE evaluates accuracy on unseen ratings. Lower RMSE values indicate more accurate predictions, and comparing the training and testing RMSE helps determine how well the recommendation system generalizes to new data.


In [58]:
# Create a mask that identifies all ratings that exist
# in the training matrix (ignoring missing ratings).
train_mask = ~train.isna()

# Select the baseline predictions corresponding only to
# the coffee and tea ratings available in the training data.
train_baseline_preds = baseline_predictions[train_mask]

# Calculate the RMSE by comparing the actual training ratings
# with the baseline predicted ratings.
# .values converts the DataFrames into NumPy arrays so they
# can be used in the RMSE calculation.
train_rmse_baseline = rmse(
    train_baseline_preds.values,
    train[train_mask].values
)

# Repeat the same process for the testing data.

# Create a mask for the ratings available in the test set.
test_mask = ~test.isna()

# Select the baseline predictions corresponding to the
# coffee and tea ratings in the testing data.
test_baseline_preds = baseline_predictions[test_mask]

# Calculate the RMSE for the unseen test ratings to evaluate
# how well the recommendation model generalizes to new data.
test_rmse_baseline = rmse(
    test_baseline_preds.values,
    test[test_mask].values
)

print(f"Baseline Training RMSE: {train_rmse_baseline}")
print(f"Baseline Testing RMSE: {test_rmse_baseline}")

Baseline Training RMSE: 1.1784744729239294
Baseline Testing RMSE: 4.092135495883129


### Summary

The raw average in this recommendation system is **6.94**, meaning most New Yorkers rate coffee and tea locations positively. The user and location biases reveal the rating behavior of New Yorkers, with some being more generous or strict depending on the location. For example, HeyTea and Tiger Sugar received higher-than-average ratings, while Blue Bottle received a lower score. The baseline model resulted in a training RMSE of **1.18**, showing strong performance on the training data. However, a much higher test RMSE of **4.09** indicates that the error increases significantly on unseen data, meaning the baseline model does not generalize well. This gap shows that the model can capture patterns in the training data but has weaker performance for new recommendations.
